**.join use for concatenation of string and " " before it add a space between two strings.**

In [ ]:
link = "https://www.youtube.com/watch?v=bMTlNeKqV4o"
link_id = link[32:43]
print(link_id)

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_core.runnables import RunnableParallel , RunnableLambda ,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [6]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [7]:
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5420.36it/s]


In [8]:
video_id = "MY5SatbZMAo"

api = YouTubeTranscriptApi()

try:
    transcript = api.fetch(video_id, languages=["en"])

    text = " ".join(item.text for item in transcript)


except Exception as e:
    print(e)

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)

chunks = splitter.create_documents([text])

In [10]:
chunks[0].page_content

'Translator: Riaki Poništ\nReviewer: Peter van de Ven Thank you so much. I am a journalist. My job is to talk to people\nfrom all walks of life, all over the world. Today, I want to tell you why I decided to do this with my life\nand what I\'ve learned. My story begins in Caracas, Venezuela, in South America, where I grew up; a place that to me was,\nand always will be, filled with magic and wonder. Frоm a very young age, my parents wanted me\nto have a wider view of the world. I remember one time\nwhen I was around seven years old, my dad came up to me and said, "Mariana, I\'m going to send you\nand your little sister..." - who was six at the time - "...to a place where nobody\nspeaks Spanish. I want you to experience\ndifferent cultures." He went on and on about the benefits\nof spending an entire summer in this summer camp in the United States, stressing a little phrase that I didn\'t pay'

In [11]:
vectors = FAISS.from_documents(chunks,embeddings)

In [12]:
vectors.index.ntotal

21

In [13]:
retriever = vectors.as_retriever(search_type="similarity",kwargs={"k":4})

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer from the below context. If the answer is not in the context,
say "I don't have enough information."

Context:
{context}

Question:
{question}


""")

In [15]:
question = "What do you make special?"
retrieved_docs = retriever.invoke(question)

def format_docs(retrieved_docs):
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return context

In [16]:
parallel_chain = RunnableParallel({"context":retriever | RunnableLambda(format_docs),"question":RunnablePassthrough()})

In [17]:
final_chain = parallel_chain | prompt | model | parser

In [19]:
answer = final_chain.invoke("What this video is about?")
print(answer)

This video is about a journalist's life journey, why she decided to pursue her profession, and what she has learned. It covers her childhood in Caracas, Venezuela, her experience attending a summer camp in Brainerd, Minnesota, where she encountered cultural differences, and her work covering the 2016 election for NBC News, including a poignant interaction with an eight-year-old girl named Angelina from an undocumented family. The story emphasizes the importance of experiencing different cultures, understanding diverse perspectives, and the necessity of dialogue.
